# Q2 — Completion Analysis

**Business Question:** What factors are associated with higher completion rates? What patterns emerge in terminated/withdrawn trials?

Only *resolved* trials (Completed + Stopped) are included in completion rate calculations. Active trials are excluded — they haven't reached an outcome yet, and including them would artificially deflate rates.

**Resolved trials:** 1,233 total (1,025 Completed + 208 Stopped). Overall resolved completion rate: **83.1%**.

---

In [14]:
# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────
# CONFIG — update username if needed
# ─────────────────────────────────────────
DB_USER = "vittoriobariosco"  # your username
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "clinical_trials" # your PostgreSQL database name 
CSV_PATH = "/Users/vittoriobariosco/Documents/APPLICATIONS/MIGx/data/processed/clinical_trials_clean.csv"  #path to cleaned CSV from EDA notebook

# ─────────────────────────────────────────
# 1. CONNECT TO POSTGRESQL
# ─────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# ── Global Plotly theme (minimal/corporate) ──────────────────────────────
COLORS = {
    'primary': '#2563EB',
    'secondary': '#64748B',
    'accent': '#F59E0B',
    'success': '#10B981',
    'danger': '#EF4444',
    'light_bg': '#F8FAFC',
    'text': '#1E293B',
    'muted': '#94A3B8',
}

PALETTE = ['#2563EB', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6',
           '#EC4899', '#14B8A6', '#F97316', '#6366F1', '#64748B']

LAYOUT_DEFAULTS = dict(
    font=dict(family='Helvetica Neue, Arial, sans-serif', color=COLORS['text']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=30, t=60, b=60),
    title_font_size=16,
    title_x=0.0,
    hoverlabel=dict(bgcolor='white', font_size=12),
)

def clean_axes(fig, show_grid_y=True):
    fig.update_xaxes(showgrid=False, linecolor='#E2E8F0', linewidth=1)
    fig.update_yaxes(
        showgrid=show_grid_y, gridcolor='#F1F5F9', gridwidth=1,
        linecolor='#E2E8F0', linewidth=1
    )
    return fig

print('✅ Setup complete')

✅ Setup complete


---
## 2A — Completion Rate by Phase & Study Type

In [15]:
# ── Query 2A ──────────────────────────────────────────────────────────────
query_2a = """
WITH resolved AS (
    SELECT study_id, phase, study_type, status_group, enrollment
    FROM studies
    WHERE status_group IN ('Completed', 'Stopped')
)
SELECT
    phase, study_type,
    COUNT(*) AS total_resolved,
    COUNT(*) FILTER (WHERE status_group = 'Completed') AS completed,
    COUNT(*) FILTER (WHERE status_group = 'Stopped') AS stopped,
    ROUND(100.0 * COUNT(*) FILTER (WHERE status_group = 'Completed') / COUNT(*), 1) AS completion_rate_pct,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY enrollment)::numeric, 0) AS median_enrollment
FROM resolved
WHERE phase != 'Unknown'
GROUP BY phase, study_type
ORDER BY completion_rate_pct DESC;
"""

df_2a = pd.read_sql(query_2a, engine)
df_2a

,phase,study_type,total_resolved,completed,stopped,completion_rate_pct,median_enrollment
0,Not Applicable,Interventional,260,226,34,86.9,72.0
1,Phase 1,Interventional,49,38,11,77.6,27.0
2,Phase 4,Interventional,36,22,14,61.1,47.0
3,Phase 2,Interventional,162,95,67,58.6,38.0
4,Phase 3,Interventional,136,76,60,55.9,126.0
5,Early Phase 1,Interventional,8,3,5,37.5,7.0


In [16]:
# ── Chart 2A: Completion rate by phase (bar + marker for enrollment) ─────
# All resolved trials are Interventional

# Custom phase order for logical progression
phase_order = ['Early Phase 1', 'Phase 1', 'Phase 2', 'Phase 3', 'Phase 4', 'Not Applicable']
df_2a_sorted = df_2a.set_index('phase').loc[phase_order].reset_index()

fig_2a = make_subplots(specs=[[{"secondary_y": True}]])

# Completion rate bars
fig_2a.add_trace(
    go.Bar(
        x=df_2a_sorted['phase'],
        y=df_2a_sorted['completion_rate_pct'],
        marker_color=[COLORS['danger'] if r < 60 else COLORS['accent'] if r < 75 else COLORS['success']
                      for r in df_2a_sorted['completion_rate_pct']],
        text=[f"{r}%<br>(n={n})" for r, n in zip(df_2a_sorted['completion_rate_pct'], df_2a_sorted['total_resolved'])],
        textposition='outside',
        textfont=dict(size=10),
        name='Completion Rate',
        showlegend=False,
    ),
    secondary_y=False,
)

# Median enrollment line
fig_2a.add_trace(
    go.Scatter(
        x=df_2a_sorted['phase'],
        y=df_2a_sorted['median_enrollment'],
        mode='lines+markers',
        line=dict(color=COLORS['secondary'], width=2, dash='dot'),
        marker=dict(size=8, color=COLORS['secondary']),
        name='Median Enrollment',
        hovertemplate='%{x}: %{y:,.0f} patients<extra></extra>',
    ),
    secondary_y=True,
)


# ── Add Legend for Bar Colors (Risk Levels) ──────────────────────────────
for label, color in [('High Risk (<60%)', COLORS['danger']), 
                     ('Moderate Risk (60-75%)', COLORS['accent']), 
                     ('Low Risk (>75%)', COLORS['success'])]:
    fig_2a.add_trace(
        go.Bar(
            x=[None], y=[None], # Invisible data
            name=label,
            marker_color=color,
            showlegend=True
        )
    )


fig_2a.update_layout(
    **LAYOUT_DEFAULTS,
    title='Completion Rate by Phase (with Median Enrollment)',
    height=450,
    width=800,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_2a.update_yaxes(title_text='Completion Rate (%)', range=[0, 105], secondary_y=False)
fig_2a.update_yaxes(title_text='Median Enrollment', secondary_y=True)
clean_axes(fig_2a)
fig_2a.show()

### Key Findings — Completion Rate by Phase

Completion rates decline as trials progress through the development pipeline:
- **Not Applicable:** 86.9% (n=260) — non-drug interventions (behavioral, diagnostic) are simplest to execute.
- **Phase 1:** 77.6% (n=49) — small, safety-focused trials with limited complexity.
- **Phase 4:** 61.1% (n=36) — post-marketing studies.
- **Phase 2:** 58.6% (n=162) — efficacy testing begins; many compounds fail here.
- **Phase 3:** 55.9% (n=136) — large, expensive confirmatory trials with the highest failure rate among established phases.
- **Early Phase 1:** 37.5% (n=8) — very small sample, but first-in-human trials are inherently high-risk.

**2. All resolved trials with known phase are Interventional.** Observational trials either have Unknown phase or are classified as Not Applicable, so a direct Interventional vs Observational comparison by phase is not possible.

---

## 2B — Completion Rate by Sponsor Type

In [17]:
# ── Query 2B ──────────────────────────────────────────────────────────────
query_2b = """
WITH resolved_sponsored AS (
    SELECT s.study_id, s.status_group, sp.agency_class
    FROM studies s
    JOIN sponsors sp ON s.study_id = sp.study_id
    WHERE s.status_group IN ('Completed', 'Stopped')
          AND sp.lead_or_collaborator = 'Lead'
)
SELECT
    agency_class,
    COUNT(*) AS total_resolved,
    COUNT(*) FILTER (WHERE status_group = 'Completed') AS completed,
    COUNT(*) FILTER (WHERE status_group = 'Stopped') AS stopped,
    ROUND(100.0 * COUNT(*) FILTER (WHERE status_group = 'Completed') / COUNT(*), 1) AS completion_rate_pct
FROM resolved_sponsored
GROUP BY agency_class
ORDER BY completion_rate_pct DESC;
"""

df_2b = pd.read_sql(query_2b, engine)
df_2b

,agency_class,total_resolved,completed,stopped,completion_rate_pct
0,Other,1058,896,162,84.7
1,Industry,162,120,42,74.1
2,NIH,10,7,3,70.0
3,U.S. Fed,3,2,1,66.7


In [18]:
# ── Chart 2B: Completion rate by sponsor type ────────────────────────────
df_2b_sorted = df_2b.sort_values('completion_rate_pct', ascending=True)

fig_2b = go.Figure()

fig_2b.add_trace(go.Bar(
    x=df_2b_sorted['completion_rate_pct'],
    y=df_2b_sorted['agency_class'],
    orientation='h',
    marker_color=COLORS['primary'],
    text=[f"{r}% (n={n})" for r, n in zip(df_2b_sorted['completion_rate_pct'], df_2b_sorted['total_resolved'])],
    textposition='outside',
    textfont=dict(size=11),
))

fig_2b.update_layout(
    **LAYOUT_DEFAULTS,
    title='Completion Rate by Lead Sponsor Type',
    xaxis_title='Completion Rate (%)',
    yaxis_title='',
    height=350,
    width=750,
    showlegend=False,
    xaxis_range=[0, 105],
)
clean_axes(fig_2b)
fig_2b.show()

### Key Findings — Completion Rate by Sponsor Type

**1. "Other" sponsors (academic institutions, hospitals, NGOs) dominate the resolved pool** with 1,058 trials and the highest completion rate (84.7%). 

**2. Industry-sponsored trials complete at 74.1% (n=162).** The 10.6 percentage-point gap vs academic sponsors is notable. 

---

## 2C — Completion Rate by Study Design Features

In [19]:
# ── Query 2C ──────────────────────────────────────────────────────────────
query_2c = """
WITH resolved_design AS (
    SELECT s.study_id, s.status_group, sd.allocation, sd.intervention_model, sd.masking
    FROM studies s
    JOIN study_design sd ON s.study_id = sd.study_id
    WHERE s.status_group IN ('Completed', 'Stopped')
)

SELECT 'Allocation' AS design_feature, allocation AS feature_value,
    COUNT(*) AS total_resolved,
    COUNT(*) FILTER (WHERE status_group = 'Completed') AS completed,
    ROUND(100.0 * COUNT(*) FILTER (WHERE status_group = 'Completed') / COUNT(*), 1) AS completion_rate_pct
FROM resolved_design WHERE allocation IS NOT NULL GROUP BY allocation

UNION ALL

SELECT 'Intervention Model', intervention_model,
    COUNT(*), COUNT(*) FILTER (WHERE status_group = 'Completed'),
    ROUND(100.0 * COUNT(*) FILTER (WHERE status_group = 'Completed') / COUNT(*), 1)
FROM resolved_design WHERE intervention_model IS NOT NULL GROUP BY intervention_model

UNION ALL

SELECT 'Masking', masking,
    COUNT(*), COUNT(*) FILTER (WHERE status_group = 'Completed'),
    ROUND(100.0 * COUNT(*) FILTER (WHERE status_group = 'Completed') / COUNT(*), 1)
FROM resolved_design WHERE masking IS NOT NULL GROUP BY masking

ORDER BY design_feature, completion_rate_pct DESC;
"""

df_2c = pd.read_sql(query_2c, engine)
df_2c

,design_feature,feature_value,total_resolved,completed,completion_rate_pct
0,Allocation,Non-Randomized,58,51,87.9
1,Allocation,N/A,121,90,74.4
2,Allocation,Randomized,472,319,67.6
3,Intervention Model,Crossover Assignment,27,26,96.3
4,Intervention Model,Single Group Assignment,137,103,75.2
5,Intervention Model,Factorial Assignment,12,9,75.0
6,Intervention Model,Parallel Assignment,455,311,68.4
7,Intervention Model,Sequential Assignment,20,11,55.0
8,Masking,"Triple (Care Provider, Investigator, Outcomes ...",3,3,100.0
9,Masking,Single (Care Provider),1,1,100.0


In [43]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Prepare Data
df_alloc = df_2c[df_2c['design_feature'] == 'Allocation'].sort_values('completion_rate_pct', ascending=True)
df_model = df_2c[df_2c['design_feature'] == 'Intervention Model'].sort_values('completion_rate_pct', ascending=True)

# Masking logic: Include all categories with n >= 10 and clean the labels
df_mask = df_2c[(df_2c['design_feature'] == 'Masking') & (df_2c['total_resolved'] >= 10)].copy()
df_mask = df_mask.sort_values('completion_rate_pct', ascending=True)
df_mask['display_label'] = df_mask['feature_value'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# Calculate proportions for uniform bar thickness
counts = [len(df_alloc), len(df_model), len(df_mask)]
total_rows = sum(counts)
row_proportions = [c / total_rows for c in counts]

# 2. Initialize Subplots
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        '<b>Allocation Method</b>', 
        '<b>Intervention Model</b>', 
        '<b>Masking Level (n ≥ 10)</b>'
    ),
    vertical_spacing=0.14, 
    row_heights=row_proportions
)

# ── Subplot 1: Allocation ──
fig.add_trace(go.Bar(
    x=df_alloc['completion_rate_pct'],
    y=df_alloc['feature_value'],
    orientation='h',
    marker_color=[COLORS['success'] if r >= 80 else COLORS['accent'] if r >= 70 else COLORS['danger'] for r in df_alloc['completion_rate_pct']],
    text=[f"{r}% (n={n})" for r, n in zip(df_alloc['completion_rate_pct'], df_alloc['total_resolved'])],
    textposition='outside',
    showlegend=False,
    width=0.7 # Standardized width
), row=1, col=1)

# ── Subplot 2: Intervention Model ──
fig.add_trace(go.Bar(
    x=df_model['completion_rate_pct'],
    y=df_model['feature_value'],
    orientation='h',
    marker_color=[COLORS['success'] if r >= 75 else COLORS['accent'] if r >= 65 else COLORS['danger'] for r in df_model['completion_rate_pct']],
    text=[f"{r}% (n={n})" for r, n in zip(df_model['completion_rate_pct'], df_model['total_resolved'])],
    textposition='outside',
    showlegend=False,
    width=0.7 # Standardized width
), row=2, col=1)

# ── Subplot 3: Masking (Unified with all n>=10 categories) ──
fig.add_trace(go.Bar(
    x=df_mask['completion_rate_pct'],
    y=df_mask['display_label'], # Using the cleaned categorical labels
    orientation='h',
    marker_color=[COLORS['success'] if r >= 75 else COLORS['accent'] if r >= 65 else COLORS['danger'] for r in df_mask['completion_rate_pct']],
    text=[f"{r}% (n={n})" for r, n in zip(df_mask['completion_rate_pct'], df_mask['total_resolved'])],
    textposition='outside',
    showlegend=False,
    width=0.7 # Standardized width to match plots above
), row=3, col=1)

# ── Add Legend for Risk Levels (Invisible traces for legend entries) ──
for label, color in [('High Risk (<65%)', COLORS['danger']), 
                     ('Moderate Risk (65-75%)', COLORS['accent']), 
                     ('Low Risk (>75%)', COLORS['success'])]:
    fig.add_trace(go.Bar(x=[None], y=[None], name=label, marker_color=color, showlegend=True))

# 3. Final Layout Formatting
fig.update_layout(**LAYOUT_DEFAULTS)

fig.update_layout(
    title_text='<b>IMPACT OF STUDY DESIGN ON COMPLETION SUCCESS</b>',
    height=1000, 
    width=950,
    bargap=0.15,      
    bargroupgap=0.1,
    # Increased left margin to 320 to accommodate "Quadruple" masking label
    margin=dict(l=320, r=80, t=120, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

# Apply common axis styling
fig.update_xaxes(title_text="Completion Rate (%)", range=[0, 115])
fig.update_yaxes(automargin=True)

clean_axes(fig)
fig.show()

### Key Findings — Study Design & Completion

**Allocation:**
- **Non-Randomized trials complete at 87.9%** (n=58) vs **Randomized at 67.6%** (n=472). The 20-point gap is the single strongest design-level predictor of completion in this dataset.
- Randomization adds operational complexity (blinding logistics, stratification, larger samples for statistical power).

**Intervention Model:**
- **Crossover design** has the highest rate (96.3%, n=27).
- **Parallel Assignment** — completes at 68.4% (n=455).
- **Sequential Assignment** is the lowest (55.0%, n=20).

**Masking:**
- A clear pattern emerges: **more blinding layers = lower completion rates.**
  - Single blinding (Participant): 89.7%
  - Double (Participant + Investigator): 73.1%
  - Quadruple: 57.1%

**Overall pattern:** Trial design complexity is inversely correlated with completion. The more rigorous the methodology (randomized, parallel, multi-blinded), the harder it is to complete.

---

## 2D — Stopped Trials Deep Dive

In [45]:
# ── Query 2D ──────────────────────────────────────────────────────────────
query_2d = """
WITH stopped_detail AS (
    SELECT
        status, phase, study_type, enrollment,
        CASE
            WHEN completion_date IS NOT NULL AND start_date IS NOT NULL
            THEN EXTRACT(DAY FROM AGE(completion_date, start_date))
        END AS duration_days
    FROM studies
    WHERE status_group = 'Stopped'
),
stopped_summary AS (
    SELECT
        status, phase,
        COUNT(*) AS n_trials,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY enrollment)::numeric, 0) AS median_enrollment,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY duration_days)::numeric, 0) AS median_duration_days
    FROM stopped_detail
    WHERE phase != 'Unknown'
    GROUP BY status, phase
)
SELECT
    status, phase, n_trials, median_enrollment, median_duration_days,
    RANK() OVER(PARTITION BY status ORDER BY n_trials DESC) AS rank_within_status
FROM stopped_summary
ORDER BY status, rank_within_status;
"""

df_2d = pd.read_sql(query_2d, engine)
df_2d

,status,phase,n_trials,median_enrollment,median_duration_days,rank_within_status
0,Suspended,Phase 3,13,210.0,15.0,1
1,Suspended,Phase 2,6,88.0,16.0,2
2,Suspended,Not Applicable,2,65.0,19.0,3
3,Suspended,Phase 4,2,1214.0,15.0,3
4,Suspended,Early Phase 1,2,300.0,9.0,3
5,Terminated,Phase 2,27,21.0,15.0,1
6,Terminated,Phase 3,21,58.0,13.0,2
7,Terminated,Not Applicable,13,37.0,10.0,3
8,Terminated,Phase 4,5,44.0,16.0,4
9,Terminated,Phase 1,4,10.0,17.0,5


In [46]:
# ── Chart 2D-1: Stopped trials heatmap — status × phase ─────────────────
# Pivot for heatmap: rows = failure type, columns = phase
phase_order = ['Early Phase 1', 'Phase 1', 'Phase 2', 'Phase 3', 'Phase 4', 'Not Applicable']
status_order = ['Withdrawn', 'Terminated', 'Suspended']

pivot = df_2d.pivot_table(
    index='status', columns='phase', values='n_trials', fill_value=0
)
# Reorder
pivot = pivot.reindex(index=status_order, columns=phase_order, fill_value=0)

fig_2d1 = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale=[[0, '#F8FAFC'], [0.5, '#93C5FD'], [1, '#1E40AF']],
    text=pivot.values,
    texttemplate='%{text}',
    textfont=dict(size=13),
    hovertemplate='%{y} — %{x}: %{z} trials<extra></extra>',
    showscale=False,
))

fig_2d1.update_layout(
    **LAYOUT_DEFAULTS,
    title='Stopped Trials: Failure Type × Phase',
    xaxis_title='Phase',
    yaxis_title='',
    height=320,
    width=750,
)
fig_2d1.show()

In [47]:
# ── Chart 2D-2: Median enrollment by failure type ────────────────────────
# Aggregate across phases to see overall pattern per failure type
df_failure_summary = df_2d.groupby('status').agg(
    total_trials=('n_trials', 'sum'),
).reset_index()

# Use the raw data for per-status median (re-aggregate from df_2d)
failure_colors = {
    'Withdrawn': COLORS['secondary'],
    'Terminated': COLORS['accent'],
    'Suspended': COLORS['danger'],
}

fig_2d2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Withdrawn (n=107)', 'Terminated (n=74)', 'Suspended (n=27)'],
    shared_yaxes=True,
)

for i, status in enumerate(['Withdrawn', 'Terminated', 'Suspended'], 1):
    subset = df_2d[df_2d['status'] == status].sort_values('n_trials', ascending=True)
    fig_2d2.add_trace(
        go.Bar(
            x=subset['n_trials'],
            y=subset['phase'],
            orientation='h',
            marker_color=failure_colors[status],
            text=subset['n_trials'],
            textposition='outside',
            textfont=dict(size=10),
            showlegend=False,
        ),
        row=1, col=i,
    )

fig_2d2.update_layout(
    **LAYOUT_DEFAULTS,
    title='Phase Distribution within Each Failure Type',
    height=400,
    width=950,
)
clean_axes(fig_2d2)
fig_2d2.show()

### Key Findings — Stopped Trials Deep Dive

**Three distinct failure modes:**

**1. Withdrawn (107 trials, 51.4% of stopped)**
- Median enrollment = **0** across all phases. These trials were canceled before enrolling a single patient.
- Phase 2 (34) and Phase 3 (26) account for 56% of withdrawals. These are trials that were designed but never operationally launched — likely due to recruitment failure, funding withdrawal, or protocol infeasibility.
- **Implication:** The problem is perhaps in the planning stage. Better feasibility assessment before registration would reduce this.

**2. Terminated (74 trials, 35.6% of stopped) — "Failed during execution"**
- Median enrollment > 0 across all phases (10–58 patients).
- Phase 2 leads (27 trials), followed by Phase 3 (21). These represent genuine scientific failures: the intervention didn't work, safety issues emerged, or enrollment was too slow to continue.

**3. Suspended (27 trials, 13.0% of stopped) — "On hold"**
- Median enrollment is the highest (88–1,214 depending on phase) — these are larger, more established trials.
- Phase 3 dominates (13 trials). The high enrollment suggests these were temporarily halted due to external factors (regulatory holds, safety reviews, supply chain issues) rather than fundamental design flaws.

---

## Summary & Recommendations

### Factors Associated with Higher Completion Rates:

1.  **Operational simplicity is the strongest predictor of success.** Trials with fewer technical hurdles complete significantly more often: Non-randomized designs (**87.9%**) outperform Randomized (**67.6%**), and Crossover models (**96.3%**) exceed Parallel models (**68.4%**). During pandemic conditions, minimizing logistical complexity is vital for reaching study endpoints.

2.  **Academic (Other) sponsors demonstrate greater agility.** Academic institutions complete **84.7%** of their trials compared to **74.1%** for Industry sponsors. This suggests that smaller, often single-site academic studies are more resilient to the operational disruptions of a pandemic than large-scale commercial efforts.

### Patterns in Stopped (Terminated/Withdrawn) Trials:

4.  **The "Zero-Patient" Feasibility Barrier:** Where **51.4%** of stopped trials are **Withdrawn** before enrolling a single participant. This indicates that failure was administrative or logistical (funding, site availability, or regulatory hurdles) rather than clinical.

5.  **The Complexity Penalty and Phase 3 Risk:** There is a direct pattern where increased scientific rigor correlates with failure. While Single-blind trials enjoy an **89.7%** completion rate, the most complex "Quadruple-blind" trials drop to **57.1%**. This pattern is most dangerous in **Phase 3**, which has the lowest completion rate (**55.9%**) despite having the highest **median enrollment (126 patients)** and highest sunk cost.

6.  **"Fail Fast" Decision Logic:** Most stopped trials reach their terminal status very quickly, with a **median duration of 0–19 days**. This pattern is actually a positive indicator of researcher responsiveness, showing that sponsors are identifying non-viable trials early and cutting losses rather than allowing resource-draining studies to persist indefinitely.

---
